# SubspaceAD (CVPR 2026): checkpointed Kaggle benchmark on MVTec-AD and VisA

SubspaceAD is a training-free few-shot anomaly detector ([paper / official code](https://github.com/CLendering/SubspaceAD)):

1. Patch features come from a frozen **`facebook/dinov2-with-registers-giant`**: `hidden_states[-12 … -18]` are averaged, and the CLS and register tokens are dropped.
2. A **PCA subspace** of the normal features is fitted, keeping 99 % of the explained variance (`--pca_ev 0.99`).
3. The **anomaly score** is the squared reconstruction residual. The patch map is upsampled bilinearly to 672 px and blurred with a 3×3 Gaussian (σ = 4, reflect-101). The image score is the mean of the top 1 % of pixels.

Two protocols are evaluated:

| Regime | PCA fitted on | Official script |
|---|---|---|
| **R** (few-shot, 1/2/4-shot × several seeds) | *k* normal training images + 30 random rotations each (black fill, nearest; no augmentation for `transistor`) | `scripts/benchmark_few_shot.sh` |
| **T** (batched zero-shot) | the whole unlabeled test set of the category (no augmentation) | `scripts/benchmark_batched0shot.sh` |

## What is cached, and why you never pay twice

| Stage | Cost | Stored under `subspacead_ckpt/<smoke|full>/` | Invalidated by |
|---|---|---|---|
| Per-image statistics of the augmented shots | GPU | `feat_<feat_id>/stats/` (kept only if `KEEP_STATS`) | backbone, layers, image size, `N_AUG`, no-aug list, dtype |
| PCA models + raw 48×48 **float32** score maps (Regime R) | GPU | `feat_<feat_id>/fit_<fit_id>/<ds>/<cat>/maps.npy` | the above + `TAU`, `SHOTS`, `SEEDS` |
| Raw score maps (Regime T) | GPU | `regT_<t_id>/<ds>/<cat>/maps.npy` | backbone, layers, image size, dtype, `TAU` |
| Test-set labels + native-resolution GT masks | CPU | `gt/<ds>/<cat>/` | dataset content (signature check) |
| Metrics + per-image scores | CPU only | `result_<eval_id>.json`, `per_item_<eval_id>.parquet` | `TOP_PCT`, blur, `EVAL_SIZE`, metric flags |

* Every cache folder is named after a hash of the settings it depends on. Every cached file also stores its own fingerprint (ids, test-set signature, shape, CRC) and is checked on load. **Files that fail the check are deleted and recomputed, never reused.**
* Changing only evaluation settings re-computes metrics from the cached maps and masks. This needs **no backbone, no forward pass and no dataset**.
* The backbone and the datasets load lazily. If everything is cached, neither is touched.
* All writes are atomic (temp file + `os.replace`). Test scoring runs in resumable chunks, and a time budget stops the run cleanly before Kaggle's 12 h limit.

## Kaggle settings

* **Accelerator:** `GPU T4` or `GPU T4 x2`. Two GPUs process two categories in parallel; one GPU also works.
* **Internet:** ON for the first session (Hugging Face weights and, if not attached, the datasets).
* **Datasets:** attach MVTec AD (and optionally VisA) with *Add Input*. Otherwise they are downloaded to `/kaggle/temp`, which is not part of the saved output.

## Workflow

1. Run with `QUICK = True` (MVTec `bottle`, 1 seed). It writes to `subspacead_ckpt/smoke/`.
2. Set `QUICK = False` (writes to `subspacead_ckpt/full/`) and use **Save Version → Save & Run All**.
3. If the time budget is reached, open a new session and use *Add Input → Notebook Output* (this notebook's previous version), then Run All. Finished work is restored and skipped.

## 1. Configuration

All settings live in `CFG`. The comment next to each key says which cache id it belongs to:

* `[feat]` changes the features. Everything downstream is recomputed.
* `[fit]` changes the PCA models and raw maps.
* `[T]` changes the Regime T maps.
* `[eval]` changes only the metrics. They are recomputed on the CPU from cached maps.
* `[run]` changes only how the work is executed. Results are never invalidated.

Advanced: the environment variables `SUBSPACEAD_WORK` (work dir) and `SUBSPACEAD_CFG` (JSON dict merged into `CFG`) override the defaults without editing the cell. The CPU test harness uses them.

In [ ]:
# ============================ 1. CONFIGURATION ============================
import hashlib
import json
import math
import os
import platform
import queue
import shutil
import subprocess
import sys
import threading
import time
import warnings
import zlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

QUICK = True  # <- smoke test first (MVTec 'bottle', 1 seed). Set False for the full benchmark.

PROTOCOL = "subspacead-final-v1"  # bump only if the pipeline logic changes in a way that invalidates old caches

CFG: Dict[str, Any] = dict(
    # ---- backbone / features ------------------------------------------------------------------
    BACKBONE="facebook/dinov2-with-registers-giant",  # [feat] official backbone (DINOv2-G with registers)
    BACKBONE_LOCAL=None,             # [run]  local HF folder with the same weights (offline use)
    DTYPE="fp16",                    # [feat] "fp16" = fp32 weights + CUDA autocast (fast on T4); "fp32" = exact official
    LAYERS=[-12, -13, -14, -15, -16, -17, -18],  # [feat] indices into hidden_states (official --layers default)
    IMG_SIZE=672,                    # [feat] input resolution, multiple of 14 (672 -> 48x48 patches)
    N_AUG=30,                        # [feat] random rotations per shot image, angles U(0, 345), black fill (official --aug_count)
    NO_AUG_CATEGORIES=["transistor"],  # [feat] official --no_aug_categories default
    # ---- PCA / protocol -----------------------------------------------------------------------
    TAU=0.99,                        # [fit][T] explained-variance threshold (official --pca_ev)
    SHOTS=[1, 2, 4],                 # [fit] k-shot settings (Regime R)
    SEEDS=[0, 1, 2, 3, 4],           # [fit] shot-sampling seeds (the official script runs one seed, 42)
    RUN_REGIME_T=True,               # [run] also evaluate batched zero-shot (PCA fitted on the test set)
    # ---- post-processing + metrics (CPU only, never needs the backbone) ------------------------
    TOP_PCT=1.0,                     # [eval] image score = mean of the top 1 % pixels of the final map (official mtop1p)
    BLUR_K=3, BLUR_SIGMA=4.0,        # [eval] official post_process_map: cv2.GaussianBlur((3, 3), 4) at IMG_SIZE
    EVAL_SIZE=256,                   # [eval] pixel-metric resolution (official = IMG_SIZE = 672; 256 is ~7x cheaper)
    PIXEL_METRICS=True,              # [eval] pixel AUROC / AP / oracle-F1 (exact, sort based)
    AUPRO=True,                      # [eval] AUPRO@30 % and AUPRO@5 % (official algorithm, 300 thresholds)
    PIXEL_METRICS_SEEDS="all",       # [eval] "all" or "first": pixel metrics for every seed or the first seed only
    STRICT_MASKS=True,               # [run]  error if an anomalous test image has no ground-truth mask
    # ---- data ---------------------------------------------------------------------------------
    DATASETS=["mvtec", "visa"],
    CATEGORIES=None,                 # e.g. {"mvtec": ["bottle", "cable"]}; None = all categories
    DATA_ROOTS=None,                 # e.g. {"mvtec": "/kaggle/input/mvtec-ad"}; None = auto-discover / download
    # ---- execution ----------------------------------------------------------------------------
    BATCH=4,                         # [run] images per forward pass (lower to 2 on CUDA OOM)
    NUM_WORKERS=0 if os.name == "nt" else 2,  # [run] DataLoader workers per GPU (2 on Linux/Kaggle; 0 avoids Windows spawn pickling)
    MAX_GPUS=2,                      # [run] categories are processed on up to this many GPUs in parallel
    CHUNK=32,                        # [run] test images per resumable checkpoint chunk
    TIME_BUDGET_MIN=11 * 60,         # [run] stop cleanly before Kaggle's 12 h session limit
    TRUNCATE_UNUSED_BLOCKS=True,     # [run] drop transformer blocks after the deepest used layer (self-checked, ~27 % faster)
    KEEP_STATS=False,                # [run] keep per-image statistics (~19 MB each) so SHOTS/SEEDS/TAU changes skip stage A
    KEEP_MODELS=False,               # [run] keep the fitted PCA models (mu, components) next to the maps
    ALLOW_CPU=False,                 # [run] allow backbone forward passes on CPU (only for tests / tiny models)
    RESTORE_FROM_INPUT=True,         # [run] copy checkpoints from a previous version attached under /kaggle/input
)
if QUICK:
    CFG.update(DATASETS=["mvtec"], CATEGORIES={"mvtec": ["bottle"]}, SEEDS=[0], SHOTS=[1, 4])
if os.environ.get("SUBSPACEAD_CFG"):
    CFG.update(json.loads(os.environ["SUBSPACEAD_CFG"]))

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path(os.environ.get("SUBSPACEAD_WORK") or ("/kaggle/working" if ON_KAGGLE else "./work")).resolve()
RUN_MODE = "smoke" if QUICK else "full"           # smoke tests never share a folder with the full run
CKPT_ROOT = WORK / "subspacead_ckpt"
CKPT = CKPT_ROOT / RUN_MODE
RESULTS_DIR = WORK / f"results_{RUN_MODE}"
DATA_CACHE = Path("/kaggle/temp/data") if ON_KAGGLE else WORK / "data"   # /kaggle/temp is not saved into the output
CKPT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
START_TIME = time.time()

assert CFG["IMG_SIZE"] % 14 == 0, "IMG_SIZE must be a multiple of the ViT patch size (14)"
assert CFG["PIXEL_METRICS_SEEDS"] in ("all", "first")
assert CFG["DTYPE"] in ("fp16", "fp32")


def _h(obj: Any) -> str:
    """Short, stable hash of a JSON-serialisable object (used for cache folder names)."""
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:10]


def backbone_fp() -> Dict[str, Any]:
    """Everything that changes the features of a (non-augmented) image."""
    return dict(protocol=PROTOCOL, backbone=CFG["BACKBONE"], layers=list(CFG["LAYERS"]),
                img=CFG["IMG_SIZE"], dtype=CFG["DTYPE"])


def feat_fp() -> Dict[str, Any]:
    """Backbone + augmentation of the shot images (Regime R statistics)."""
    return dict(backbone=_h(backbone_fp()), n_aug=CFG["N_AUG"],
                no_aug=sorted(CFG["NO_AUG_CATEGORIES"]), aug="pil-rotate-nearest-black-U(0,345)")


def fit_fp() -> Dict[str, Any]:
    """Features + everything that changes the Regime R PCA models / raw maps."""
    return dict(feat=feat_id(), tau=CFG["TAU"], shots=list(CFG["SHOTS"]), seeds=list(CFG["SEEDS"]),
                sampling="perm(RandomState(1000+seed))[:k]")


def t_fp() -> Dict[str, Any]:
    """Everything that changes the Regime T (batched zero-shot) maps."""
    return dict(backbone=_h(backbone_fp()), tau=CFG["TAU"], regime="T")


def eval_fp() -> Dict[str, Any]:
    """Post-processing + metrics only (never needs a forward pass)."""
    return dict(metrics="v1", top=CFG["TOP_PCT"], bk=CFG["BLUR_K"], bs=CFG["BLUR_SIGMA"], ev=CFG["EVAL_SIZE"],
                pix=CFG["PIXEL_METRICS"], pro=CFG["AUPRO"], pix_seeds=CFG["PIXEL_METRICS_SEEDS"])


def feat_id() -> str:
    return _h(feat_fp())


def fit_id() -> str:
    return _h(fit_fp())


def t_id() -> str:
    return _h(t_fp())


def eval_id() -> str:
    return _h(eval_fp())


def grid() -> int:
    """Patch-grid side length (672 / 14 = 48)."""
    return CFG["IMG_SIZE"] // 14


def stats_dir(ds: str, cat: str) -> Path:
    return CKPT / f"feat_{feat_id()}" / "stats" / ds / cat


def fit_dir(ds: str, cat: str) -> Path:
    return CKPT / f"feat_{feat_id()}" / f"fit_{fit_id()}" / ds / cat


def t_dir(ds: str, cat: str) -> Path:
    return CKPT / f"regT_{t_id()}" / ds / cat


def gt_dir(ds: str, cat: str) -> Path:
    return CKPT / "gt" / ds / cat


def regime_dir(regime: str, ds: str, cat: str) -> Path:
    return fit_dir(ds, cat) if regime == "R" else t_dir(ds, cat)


def regime_id(regime: str) -> str:
    return fit_id() if regime == "R" else t_id()


def active_regimes() -> List[str]:
    return ["R", "T"] if CFG["RUN_REGIME_T"] else ["R"]


print(f"QUICK={QUICK} | mode={RUN_MODE} | ON_KAGGLE={ON_KAGGLE}\nCKPT={CKPT}\nDATA_CACHE={DATA_CACHE}")
print(f"cache ids: feat={feat_id()} fit={fit_id()} T={t_id()} eval={eval_id()}")

## 2. Utilities: atomic I/O, logging, time budget

Every checkpoint is written to a temporary file and then moved into place with `os.replace`, so the move is atomic. A crash or a Kaggle timeout can never leave a half-written file that looks valid. Logging is thread-safe because the optional 2-GPU mode runs one worker thread per GPU.

In [ ]:
# ============================ 2. UTILITIES ============================
STOP_EVENT = threading.Event()   # set when the time budget is reached or a worker fails
_PRINT_LOCK = threading.Lock()


def log(msg: str) -> None:
    """Thread-safe print."""
    with _PRINT_LOCK:
        print(msg, flush=True)


class TimeBudgetExceeded(Exception):
    """Raised between units of GPU work once CFG['TIME_BUDGET_MIN'] is used up."""


def check_time() -> None:
    """Stop cleanly (all finished work is already on disk) when the time budget is exhausted."""
    if STOP_EVENT.is_set() or (time.time() - START_TIME) / 60.0 > CFG["TIME_BUDGET_MIN"]:
        STOP_EVENT.set()
        raise TimeBudgetExceeded()


def _tmp(path: Path) -> Path:
    # unique per thread so two workers can never collide on the same temp file
    return path.with_name(f"{path.name}.{os.getpid()}.{threading.get_ident()}.tmp")


def atomic_save_npy(path: Path, arr: np.ndarray) -> None:
    tmp = _tmp(path)
    with open(tmp, "wb") as f:
        np.save(f, arr)
    os.replace(tmp, path)


def atomic_save_npz(path: Path, **arrays: np.ndarray) -> None:
    tmp = _tmp(path)
    with open(tmp, "wb") as f:
        np.savez_compressed(f, **arrays)
    os.replace(tmp, path)


def atomic_write_json(path: Path, obj: Any) -> None:
    tmp = _tmp(path)
    tmp.write_text(json.dumps(obj, indent=1, default=str))
    os.replace(tmp, path)


def atomic_save_table(path: Path, df: pd.DataFrame) -> Path:
    """Write a parquet file (CSV fallback if no parquet engine is installed). Returns the written path."""
    tmp = _tmp(path)
    try:
        df.to_parquet(tmp, index=False)
        os.replace(tmp, path)
        return path
    except ImportError:
        tmp.unlink(missing_ok=True)
        csv_path = path.with_suffix(".csv")
        tmp = _tmp(csv_path)
        df.to_csv(tmp, index=False)
        os.replace(tmp, csv_path)
        return csv_path


def read_json(path: Path) -> Optional[Dict[str, Any]]:
    """Return the parsed JSON, or None if missing / unreadable."""
    try:
        return json.loads(Path(path).read_text())
    except Exception:
        return None


def file_crc(path: Path) -> str:
    """CRC32 of a file's bytes (streamed). Detects truncated or corrupted checkpoint copies."""
    crc = 0
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            crc = zlib.crc32(block, crc)
    return f"{crc:08x}"


def array_crc(arr: np.ndarray) -> str:
    """CRC32 of an array's raw data (used to validate cached score maps after loading)."""
    return f"{zlib.crc32(np.ascontiguousarray(arr).view(np.uint8).ravel()):08x}"


def remove(path: Path) -> None:
    """Delete a file or folder if it exists."""
    path = Path(path)
    if path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
    else:
        path.unlink(missing_ok=True)


def ensure_fingerprint(directory: Path, fp: Dict[str, Any]) -> None:
    """Write a human-readable fingerprint.json into a hash-named cache folder (and verify an existing one)."""
    directory.mkdir(parents=True, exist_ok=True)
    f = directory / "fingerprint.json"
    old = read_json(f)
    if old is None:
        atomic_write_json(f, fp)
    elif json.loads(json.dumps(old, default=str)) != json.loads(json.dumps(fp, default=str)):
        raise RuntimeError(f"hash collision / tampered cache folder: {directory}")

## 3. Datasets: discovery, loading, ground-truth cache

* **MVTec-AD:** `root/<cat>/{train/good, test/<defect>, ground_truth/<defect>}`.
* **VisA:** either the raw release (`split_csv/1cls.csv`) or the official `prepare_visa.py` layout, which looks like MVTec.
* Roots are found under `/kaggle/input`, `/kaggle/temp/data`, the work dir and `./data`, or set explicitly with `CFG["DATA_ROOTS"]`. If a dataset is missing, it is downloaded to `/kaggle/temp/data`, which is not saved into the notebook output.
* `STRICT_MASKS=True` stops with an error if an anomalous test image has no mask. With `False`, the mask is treated as empty and a warning is printed.
* **Ground-truth cache:** the first time a category is processed, its labels and **native-resolution** binary masks are saved (compressed, a few MB per category). Later metric-only runs therefore never need the dataset. Resizing a cached binary mask with NEAREST gives exactly the pixels you get by resizing the original mask file.

In [ ]:
# ============================ 3. DATASETS ============================
from PIL import Image

MVTEC_CATS = ["bottle", "cable", "capsule", "carpet", "grid", "hazelnut", "leather", "metal_nut", "pill",
              "screw", "tile", "toothbrush", "transistor", "wood", "zipper"]
VISA_CATS = ["candle", "capsules", "cashew", "chewinggum", "fryum", "macaroni1", "macaroni2",
             "pcb1", "pcb2", "pcb3", "pcb4", "pipe_fryum"]
CATS = {"mvtec": MVTEC_CATS, "visa": VISA_CATS}
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
TestItem = Tuple[Path, int, Optional[Path]]   # (image path, label 0/1, mask path or None)


def categories_of(ds: str) -> List[str]:
    return list((CFG["CATEGORIES"] or {}).get(ds) or CATS[ds])


def list_images(d: Path) -> List[Path]:
    d = Path(d)
    if not d.is_dir():
        return []
    return sorted(p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXT)


def find_root(name: str, search: Optional[Sequence[str]] = None, max_depth: int = 5) -> Optional[Path]:
    """Locate a dataset root (the folder that contains the category folders, or VisA's split_csv/)."""
    explicit = (CFG["DATA_ROOTS"] or {}).get(name)
    if explicit and search is None:
        return Path(explicit)
    cats = CATS[name]
    need = 10 if name == "mvtec" else 8
    search = search or ["/kaggle/input", str(DATA_CACHE), str(WORK), "./data"]
    for base in search:
        if not os.path.isdir(base):
            continue
        base_depth = base.rstrip(os.sep).count(os.sep)
        for root, dirs, _ in os.walk(base):
            dirs[:] = [d for d in dirs if d != "subspacead_ckpt"]          # never mistake checkpoints for data
            if root.count(os.sep) - base_depth >= max_depth:
                dirs[:] = []
                continue
            if name == "visa" and "split_csv" in dirs and (Path(root) / "split_csv" / "1cls.csv").exists():
                return Path(root)
            if sum(c in dirs for c in cats) >= need:
                return Path(root)
    return None


def _download(url: str, dst: Path) -> None:
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", "-O", str(dst), url], check=True)
    else:
        import urllib.request
        urllib.request.urlretrieve(url, dst)


def ensure_dataset(name: str) -> Path:
    """Return the dataset root, downloading into DATA_CACHE (/kaggle/temp/data on Kaggle) if necessary."""
    root = find_root(name)
    if root is not None:
        log(f"[{name}] found at {root}")
        return root
    log(f"[{name}] not attached -> downloading into {DATA_CACHE} (needs Internet ON)")
    DATA_CACHE.mkdir(parents=True, exist_ok=True)
    if name == "mvtec":
        os.environ.setdefault("KAGGLEHUB_CACHE", str(DATA_CACHE / "kagglehub"))
        try:
            import kagglehub
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
            import kagglehub
        # NOTE: community mirror, slug not verified by us; if it fails use Add Input -> search "MVTec AD"
        p = kagglehub.dataset_download("ipythonx/mvtec-ad")
        root = find_root("mvtec", search=[str(p)])
    else:
        out = DATA_CACHE / "VisA"
        if not (out / ".extracted").exists():
            tar = DATA_CACHE / "VisA_20220922.tar"
            if not tar.exists():
                _download("https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar", tar)
            out.mkdir(parents=True, exist_ok=True)
            subprocess.run(["tar", "-xf", str(tar), "-C", str(out)], check=True)
            (out / ".extracted").touch()
            tar.unlink(missing_ok=True)
        root = find_root("visa", search=[str(out)])
    if root is None:
        raise RuntimeError(f"Could not locate the {name} dataset. Attach it via 'Add Input' and re-run.")
    log(f"[{name}] ready at {root}")
    return root


def _load_mvtec_like(root: Path, cat: str) -> Tuple[List[Path], List[TestItem]]:
    """root/cat/{train/good, test/<defect>, ground_truth/<defect>} (MVTec-AD and prepared VisA)."""
    base = Path(root) / cat
    train = list_images(base / "train" / "good")
    test: List[TestItem] = []
    tdir = base / "test"
    if not tdir.is_dir():
        raise FileNotFoundError(f"missing test folder: {tdir}")
    for ddir in sorted(p for p in tdir.iterdir() if p.is_dir()):
        for p in list_images(ddir):
            if ddir.name.lower() in ("good", "normal"):
                test.append((p, 0, None))
            else:
                gt = base / "ground_truth" / ddir.name
                cands = [gt / f"{p.stem}_mask.png", gt / f"{p.stem}.png", gt / f"{p.stem}_mask.jpg", gt / f"{p.stem}.jpg"]
                test.append((p, 1, next((c for c in cands if c.exists()), None)))
    return train, test


def _load_visa_csv(root: Path, cat: str) -> Tuple[List[Path], List[TestItem]]:
    """Raw VisA release + split_csv/1cls.csv (columns: object, split, label, image, mask)."""
    root = Path(root)
    df = pd.read_csv(root / "split_csv" / "1cls.csv")
    df.columns = [c.strip().lower() for c in df.columns]
    d = df[df["object"] == cat]
    train = sorted(root / r for r in d[d["split"] == "train"]["image"])
    test: List[TestItem] = []
    for r in d[d["split"] == "test"].sort_values("image").itertuples():
        if str(r.label).strip().lower() == "normal":
            test.append((root / r.image, 0, None))
        else:
            m = root / r.mask if isinstance(r.mask, str) and r.mask.strip() else None
            test.append((root / r.image, 1, m if (m is not None and m.exists()) else None))
    return train, test


def rel(p: Path, root: Path) -> str:
    """Portable path relative to the dataset root (independent of where the dataset is mounted)."""
    return Path(os.path.relpath(str(p), str(root))).as_posix()


class Category:
    """File lists of one dataset category plus the signatures used to validate cached results."""

    def __init__(self, ds: str, cat: str, root: Path):
        self.ds, self.cat, self.root = ds, cat, Path(root)
        if ds == "visa" and (self.root / "split_csv" / "1cls.csv").exists():
            self.train, self.test = _load_visa_csv(self.root, cat)
        else:
            self.train, self.test = _load_mvtec_like(self.root, cat)
        if not self.train or not self.test:
            raise FileNotFoundError(f"{ds}/{cat}: empty train ({len(self.train)}) or test ({len(self.test)}) split")
        missing = [str(p) for p, y, m in self.test if y == 1 and m is None]
        if missing:
            msg = f"{ds}/{cat}: {len(missing)} anomalous test images have no mask (first: {missing[0]})"
            if CFG["STRICT_MASKS"]:
                raise FileNotFoundError(msg + "  (set CFG['STRICT_MASKS']=False to treat them as empty masks)")
            log(f"  [warn] {msg} -> treated as empty masks")
        self.rel_train = [rel(p, self.root) for p in self.train]
        self.rel_test = [rel(p, self.root) for p, _, _ in self.test]
        self.rel_masks = [rel(m, self.root) if m is not None else "" for _, _, m in self.test]
        self.labels = np.array([y for _, y, _ in self.test], dtype=np.int64)
        self.test_sig = _h([self.rel_test, self.labels.tolist(), self.rel_masks])
        self.train_sig = _h(self.rel_train)

    @property
    def test_paths(self) -> List[Path]:
        return [p for p, _, _ in self.test]


def load_mask_native(path: Optional[Path]) -> Optional[np.ndarray]:
    return None if path is None else (np.asarray(Image.open(path).convert("L")) > 0)


def resize_mask(mask: Optional[np.ndarray], size: int) -> np.ndarray:
    """NEAREST resize of a binary mask (identical to resizing the original mask file with NEAREST)."""
    if mask is None:
        return np.zeros((size, size), bool)
    return np.asarray(Image.fromarray(mask.astype(np.uint8) * 255).resize((size, size), Image.NEAREST)) > 0


def save_gt_cache(c: Category) -> Dict[str, Any]:
    """Store labels + native binary masks of the test set, so metrics never need the dataset again."""
    d = gt_dir(c.ds, c.cat)
    old = load_gt_cache(c.ds, c.cat)
    if old is not None and old["test_sig"] == c.test_sig:
        return old
    d.mkdir(parents=True, exist_ok=True)
    arrays = {f"m{i}": load_mask_native(m) for i, (_, _, m) in enumerate(c.test) if m is not None}
    atomic_save_npz(d / "masks.npz", **arrays)
    meta = dict(test_sig=c.test_sig, train_sig=c.train_sig, n_train=len(c.train), rel_paths=c.rel_test,
                rel_masks=c.rel_masks, labels=c.labels.tolist(), masks_crc=file_crc(d / "masks.npz"))
    atomic_write_json(d / "testset.json", meta)       # written last = completion marker
    return meta


def load_gt_cache(ds: str, cat: str) -> Optional[Dict[str, Any]]:
    """Validated ground-truth cache (None if missing or corrupted)."""
    d = gt_dir(ds, cat)
    meta = read_json(d / "testset.json")
    if meta is None or not (d / "masks.npz").exists() or file_crc(d / "masks.npz") != meta.get("masks_crc"):
        return None
    return meta


def gt_masks(ds: str, cat: str, size: int) -> np.ndarray:
    """(n_test, size, size) bool masks from the ground-truth cache."""
    meta = load_gt_cache(ds, cat)
    assert meta is not None, f"{ds}/{cat}: ground-truth cache missing"
    with np.load(gt_dir(ds, cat) / "masks.npz") as z:
        return np.stack([resize_mask(z[f"m{i}"] if f"m{i}" in z.files else None, size)
                         for i in range(len(meta["labels"]))])

## 4. Metrics with a built-in self-test

| Level | Metrics |
|---|---|
| Image | AUROC, AP, oracle max-F1, TPR @ FPR 1 % and 5 % |
| Pixel (`PIXEL_METRICS`) | AUROC, AP, oracle max-F1 |
| Region (`AUPRO`) | AUPRO @ 30 % and @ 5 % FPR, using the official `compute_aupro` algorithm (300 FPR-linear thresholds, 8-connectivity, (0,0) anchor, interpolated at the limit) |

**CPU efficiency.** Each map set needs only two sorts: anomalous pixels and normal pixels. Pixel AUROC (Mann-Whitney with ties), AP, max-F1 and both AUPRO limits are then computed **exactly** from these sorted arrays with `searchsorted`. This avoids sklearn's slow full-array `roc_auc_score` / `precision_recall_curve` on tens of millions of pixels. With `PIXEL_METRICS_SEEDS="first"`, pixel metrics are computed for only one seed per shot. The self-test below checks the fast formulas against sklearn.

In [ ]:
# ============================ 4. METRICS ============================
from scipy import ndimage
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve

_trapz = getattr(np, "trapezoid", None) or np.trapz
NAN = float("nan")
IMAGE_KEYS = ["img_auroc", "img_ap", "img_f1_max_oracle", "img_tpr_at_fpr1", "img_tpr_at_fpr5"]
PIXEL_KEYS = ["pix_auroc", "pix_ap", "pix_f1_max_oracle"]
PRO_LIMITS = {"aupro_30": 0.30, "aupro_5": 0.05}
METRIC_KEYS = IMAGE_KEYS + PIXEL_KEYS + list(PRO_LIMITS)


def rank_metrics(pos_sorted: np.ndarray, neg_sorted: np.ndarray) -> Tuple[float, float, float]:
    """Exact AUROC (ties count 1/2), average precision and oracle max-F1 from sorted positive/negative scores.

    Identical to sklearn's roc_auc_score / average_precision_score / max F1 over precision_recall_curve,
    but O((P + N) log N) with no (P + N)-sized temporary curves.
    """
    P, N = len(pos_sorted), len(neg_sorted)
    if P == 0 or N == 0:
        return NAN, NAN, NAN
    lo = np.searchsorted(neg_sorted, pos_sorted, side="left").sum(dtype=np.float64)
    hi = np.searchsorted(neg_sorted, pos_sorted, side="right").sum(dtype=np.float64)
    auroc = (lo + hi) / (2.0 * P * N)
    thr, first = np.unique(pos_sorted, return_index=True)          # recall only changes at positive scores
    n_at = np.diff(np.append(first, P)).astype(np.float64)
    tp = (P - first).astype(np.float64)                               # #pos >= t
    fp = (N - np.searchsorted(neg_sorted, thr, side="left")).astype(np.float64)   # #neg >= t
    ap = float(np.sum(n_at / P * tp / (tp + fp)))
    f1 = float(np.max(2.0 * tp / (tp + fp + P)))
    return float(auroc), ap, f1


def image_metrics(labels: np.ndarray, scores: np.ndarray) -> Dict[str, float]:
    """Image-level AUROC, AP, oracle max-F1 and TPR at FPR <= 1 % / 5 %."""
    y = np.asarray(labels).astype(int).ravel()
    s = np.asarray(scores, dtype=np.float64).ravel()
    if not 0 < y.sum() < y.size:
        return {k: NAN for k in IMAGE_KEYS}
    prec, recall, _ = precision_recall_curve(y, s)
    f1 = np.where(prec + recall > 0, 2 * prec * recall / np.maximum(prec + recall, 1e-12), 0.0)
    fpr, tpr, _ = roc_curve(y, s)
    return dict(img_auroc=float(roc_auc_score(y, s)), img_ap=float(average_precision_score(y, s)),
                img_f1_max_oracle=float(f1.max()),
                img_tpr_at_fpr1=float(tpr[fpr <= 0.01].max()), img_tpr_at_fpr5=float(tpr[fpr <= 0.05].max()))


def _region_scores(maps: np.ndarray, masks: np.ndarray) -> List[np.ndarray]:
    """Sorted scores of every 8-connected ground-truth region (official compute_aupro, step 1)."""
    structure = ndimage.generate_binary_structure(2, 2)
    out: List[np.ndarray] = []
    for i in np.flatnonzero(masks.reshape(len(masks), -1).any(1)):
        lab, n = ndimage.label(masks[i], structure=structure)
        sel = lab > 0
        ids, vals = lab[sel], maps[i][sel]
        order = np.lexsort((vals, ids))
        bounds = np.cumsum(np.bincount(ids, minlength=n + 1)[1:])
        out.extend(np.split(vals[order], bounds[:-1]))
    return out


def aupro_from_sorted(neg_sorted: np.ndarray, regions: List[np.ndarray], fpr_limit: float,
                      num_thresholds: int = 300) -> float:
    """Exact port of the official compute_aupro (main.py), reusing pre-sorted negatives and regions."""
    if not regions or neg_sorted.size == 0:
        return NAN
    n_neg = neg_sorted.size
    target = np.linspace(0.0, fpr_limit, num_thresholds + 1)[1:]
    q_idx = np.clip(np.floor((1.0 - target) * (n_neg - 1)).astype(np.int64), 0, n_neg - 1)
    thr = neg_sorted[q_idx]
    fprs = (n_neg - np.searchsorted(neg_sorted, thr, side="left")).astype(np.float64) / n_neg
    pros = np.zeros(num_thresholds, dtype=np.float64)
    for rs in regions:
        pros += (rs.size - np.searchsorted(rs, thr, side="left")) / rs.size
    pros /= len(regions)
    order = np.argsort(fprs, kind="stable")
    f = np.concatenate([[0.0], fprs[order]])
    p = np.concatenate([[0.0], pros[order]])
    if f[-1] > fpr_limit:
        cut = np.searchsorted(f, fpr_limit, side="right")
        f0, f1, p0, p1 = f[cut - 1], f[cut], p[cut - 1], p[cut]
        p_at = p0 + (p1 - p0) * (fpr_limit - f0) / (f1 - f0) if f1 > f0 else p0
        f, p = np.concatenate([f[:cut], [fpr_limit]]), np.concatenate([p[:cut], [p_at]])
    elif f[-1] < fpr_limit:
        f, p = np.concatenate([f, [fpr_limit]]), np.concatenate([p, [p[-1]]])
    return float(_trapz(p, f) / fpr_limit)


def pixel_metrics(maps: np.ndarray, masks: np.ndarray, basic: bool = True, pro: bool = True) -> Dict[str, float]:
    """Pixel AUROC / AP / max-F1 and AUPRO@30/@5 of (n, H, W) maps against (n, H, W) boolean masks."""
    out = {k: NAN for k in PIXEL_KEYS + list(PRO_LIMITS)}
    if not (basic or pro) or not masks.any() or masks.all() or not np.isfinite(maps).all():
        return out
    neg_sorted = np.sort(maps[~masks].astype(np.float32, copy=False))
    if basic:
        out["pix_auroc"], out["pix_ap"], out["pix_f1_max_oracle"] = rank_metrics(np.sort(maps[masks]), neg_sorted)
    if pro:
        regions = _region_scores(maps, masks)
        for key, lim in PRO_LIMITS.items():
            out[key] = aupro_from_sorted(neg_sorted, regions, lim)
    return out


def _metrics_self_test() -> None:
    """Fast formulas == sklearn on tied random data; AUPRO == 1 for a perfect map and ~0.5 for half the regions."""
    rng = np.random.RandomState(0)
    for _ in range(5):
        y = rng.rand(4000) < 0.1
        s = np.round(rng.randn(4000) + 1.5 * y, 1)                    # rounding creates many ties
        a, ap, f1 = rank_metrics(np.sort(s[y]), np.sort(s[~y]))
        pr, rc, _ = precision_recall_curve(y, s)
        f1_ref = np.max(np.where(pr + rc > 0, 2 * pr * rc / np.maximum(pr + rc, 1e-12), 0))
        assert abs(a - roc_auc_score(y, s)) < 1e-12, "AUROC mismatch"
        assert abs(ap - average_precision_score(y, s)) < 1e-12, "AP mismatch"
        assert abs(f1 - f1_ref) < 1e-12, "F1 mismatch"
    m = np.zeros((2, 64, 64), bool)
    m[0, 4:16, 4:16] = True
    m[1, 32:48, 32:48] = True
    noise = rng.uniform(0, 1e-3, m.shape).astype(np.float32)
    perfect = np.where(m, 1.0, 0.0).astype(np.float32) + noise
    for key, lim in PRO_LIMITS.items():
        v = pixel_metrics(perfect, m)[key]
        assert abs(v - 1.0) < 5e-3, f"perfect {key} = {v}"
    half = noise.copy()
    half[0][m[0]] = 1.0                                               # region 1 always found
    half[1][m[1]] = 0.0                                               # region 2 below every normal pixel: never found
    assert abs(pixel_metrics(half, m)["aupro_30"] - 0.5) < 0.02, "half-hit AUPRO failed"
    print("[ok] metric self-tests passed (AUROC/AP/F1 == sklearn, AUPRO sanity)")


_metrics_self_test()

## 5. Backbone, augmentation, PCA, scoring and post-processing (official formulas)

| Step | Official code | Here |
|---|---|---|
| Pre-processing | HF `AutoImageProcessor`: PIL bicubic resize to 672×672, ImageNet mean/std | PIL bicubic resize, same normalisation |
| Augmentation | `T.RandomRotation((0, 345))` on the PIL image *before* resizing (nearest, black fill) | `PIL.Image.rotate(angle, NEAREST, fillcolor=0)` before resizing, angles drawn per image from a seed derived from its relative path |
| Features | mean of `hidden_states[l]` for `l` in −12…−18, drop `1 + num_register_tokens` tokens | identical. Optionally, blocks after the deepest used layer are removed; the features are unchanged and a runtime self-check confirms it |
| PCA | float64 mean, covariance /(N−1), `eigh`, k = `searchsorted(cumvar, 0.99) + 1` | identical, built from summed sufficient statistics (Σx, Σxxᵀ, n) so per-image work can be cached |
| Score | ‖x − μ − CCᵀ(x − μ)‖² in float32 | identical |
| Map | `cv2.resize` bilinear to 672, `cv2.GaussianBlur((3,3), 4)` (reflect-101), image score = mean of the top 1 % | identical in torch (the CPU harness checks it against cv2) |

`DTYPE="fp16"` keeps the weights in fp32 and runs the ViT under CUDA autocast. LayerNorm and softmax therefore stay in fp32, which avoids the overflow risk of pure-fp16 weights. `DTYPE="fp32"` is bit-for-bit the official precision.

In [ ]:
# ============================ 5. TORCH: BACKBONE, PCA, SCORING, POST-PROCESSING ============================
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

N_CUDA = torch.cuda.device_count() if torch.cuda.is_available() else 0
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def default_device() -> torch.device:
    return torch.device("cuda:0" if N_CUDA else "cpu")


# ---------------------------------------------------------------- data loading
def load_view(path: Path, size: int, angle: Optional[float] = None) -> np.ndarray:
    """(3, size, size) uint8. Optional rotation of the ORIGINAL image first (official RandomRotation semantics)."""
    img = Image.open(path).convert("RGB")
    if angle is not None:
        img = img.rotate(float(angle), resample=Image.NEAREST, expand=False, fillcolor=(0, 0, 0))
    img = img.resize((size, size), Image.BICUBIC)
    return np.ascontiguousarray(np.asarray(img).transpose(2, 0, 1))


class ViewDataset(Dataset):
    """Items are (path, angle-or-None); returns uint8 tensors so worker -> GPU transfer is 4x smaller."""

    def __init__(self, items: Sequence[Tuple[Path, Optional[float]]], size: int):
        self.items, self.size = list(items), int(size)

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, i: int) -> torch.Tensor:
        p, a = self.items[i]
        return torch.from_numpy(load_view(p, self.size, a))


def make_loader(dataset: Dataset, batches: List[List[int]], dev: torch.device) -> DataLoader:
    """DataLoader over explicit index batches (a batch never crosses an image / chunk boundary)."""
    nw = min(int(CFG["NUM_WORKERS"]), max(1, len(batches)))
    return DataLoader(dataset, batch_sampler=batches, num_workers=nw, pin_memory=dev.type == "cuda",
                      prefetch_factor=4 if nw > 0 else None)


def rotation_angles(key: str, n_aug: int) -> List[Optional[float]]:
    """Original view + n_aug angles ~ U(0, 345), reproducible from the image's relative path."""
    rng = np.random.RandomState(zlib.crc32(key.encode()) % (2 ** 32))
    return [None] + [float(a) for a in rng.uniform(0.0, 345.0, n_aug)]


# ---------------------------------------------------------------- backbone (lazy, one copy per device)
_MODELS: Dict[str, torch.nn.Module] = {}
_MODEL_LOCK = threading.Lock()


def _raw_features(model: torch.nn.Module, x01: torch.Tensor) -> torch.Tensor:
    """x01: (B, 3, H, W) float in [0, 1] on the model device -> (B, n_patches, D) float32 averaged layer features."""
    mean = torch.tensor(IMAGENET_MEAN, device=x01.device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=x01.device).view(1, 3, 1, 1)
    x = (x01 - mean) / std
    use_amp = CFG["DTYPE"] == "fp16" and x.device.type == "cuda"
    with torch.inference_mode(), torch.autocast(device_type=x.device.type, dtype=torch.float16, enabled=use_amp):
        hs = model(pixel_values=x, output_hidden_states=True).hidden_states
    return torch.stack([hs[i].float() for i in model.sa_idx]).mean(0)[:, model.sa_drop:, :]


def prepare_backbone(model: torch.nn.Module, dev: torch.device) -> torch.nn.Module:
    """Move to device (fp32 weights), freeze, resolve layer indices and (optionally) drop unused trailing blocks."""
    model = model.to(dev).float().eval()
    for p in model.parameters():
        p.requires_grad_(False)
    cfg = model.config
    assert getattr(cfg, "patch_size", 14) == 14, "grid() assumes a patch size of 14"
    n = int(cfg.num_hidden_layers)
    idx = [(n + 1 + l) if l < 0 else l for l in CFG["LAYERS"]]
    assert all(0 < i <= n for i in idx), f"layer indices {idx} invalid for a {n}-block model"
    model.sa_idx = idx
    model.sa_drop = 1 + int(getattr(cfg, "num_register_tokens", 0) or 0)   # CLS + register tokens
    need = max(idx)
    layers = getattr(getattr(model, "encoder", None), "layer", None)
    if CFG["TRUNCATE_UNUSED_BLOCKS"] and isinstance(layers, torch.nn.ModuleList) and need < len(layers):
        probe = torch.rand(1, 3, 112, 112, generator=torch.Generator().manual_seed(0)).to(dev)
        ref = _raw_features(model, probe)
        model.encoder.layer = torch.nn.ModuleList(list(layers)[:need])
        if torch.allclose(_raw_features(model, probe), ref, rtol=1e-4, atol=1e-4):
            log(f"[{dev}] truncated backbone to {need}/{n} blocks (self-check passed: identical features)")
        else:
            model.encoder.layer = layers
            log(f"[{dev}] [warn] truncation self-check failed -> using all {n} blocks")
    log(f"[{dev}] backbone ready | blocks={n} dim={cfg.hidden_size} registers={model.sa_drop - 1} "
        f"| averaging hidden_states{list(CFG['LAYERS'])} (= {idx}) | dtype={CFG['DTYPE']}")
    return model


def load_backbone(dev: torch.device) -> torch.nn.Module:
    """Download (first time) and load the Hugging Face backbone onto `dev`."""
    from transformers import AutoModel
    src = CFG["BACKBONE_LOCAL"] or CFG["BACKBONE"]
    try:
        m = AutoModel.from_pretrained(src, attn_implementation="sdpa")
    except Exception:
        m = AutoModel.from_pretrained(src)
    return prepare_backbone(m, dev)


def get_model(dev: torch.device) -> torch.nn.Module:
    """Lazy: the backbone is only downloaded / loaded when a forward pass is really needed."""
    key = str(dev)
    with _MODEL_LOCK:                                   # serialises downloads and RAM-heavy loading
        if key not in _MODELS:
            if dev.type != "cuda" and not CFG["ALLOW_CPU"]:
                raise RuntimeError("A backbone forward pass is needed but no GPU is available. "
                                   "Notebook settings -> Accelerator -> GPU T4 (or set CFG['ALLOW_CPU']=True).")
            _MODELS[key] = load_backbone(dev)
        return _MODELS[key]


def release_model(dev: torch.device) -> None:
    with _MODEL_LOCK:
        _MODELS.pop(str(dev), None)
    if dev.type == "cuda":
        torch.cuda.empty_cache()


def extract(model: torch.nn.Module, x_uint8: torch.Tensor, dev: torch.device) -> torch.Tensor:
    """(B, 3, S, S) uint8 -> (B, grid()**2, D) float32 patch features."""
    x = x_uint8.to(dev, non_blocking=True).float().div_(255.0)
    f = _raw_features(model, x)
    assert f.shape[1] == grid() ** 2, f"token count {f.shape[1]} != {grid() ** 2}"
    if not torch.isfinite(f).all():
        raise RuntimeError("Non-finite features. Set CFG['DTYPE']='fp32' (and lower BATCH).")
    return f


# ---------------------------------------------------------------- PCA + scoring
def fit_pca(S1: torch.Tensor, S2: torch.Tensor, n: int, tau: float,
            dev: torch.device) -> Tuple[torch.Tensor, torch.Tensor, int]:
    """Official PCAModel from sufficient statistics. Returns (mu (D,), components (D, k), k) as float32 on CPU."""
    S1, S2 = S1.to(dev, torch.float64), S2.to(dev, torch.float64)
    mu = S1 / n
    cov = (S2 - n * torch.outer(mu, mu)) / max(n - 1, 1)    # == sum((x - mu)(x - mu)^T) / (N - 1)
    cov = (cov + cov.T) / 2
    evals, evecs = torch.linalg.eigh(cov)
    order = torch.argsort(evals, descending=True)
    evals, evecs = evals[order].clamp_min(0.0), evecs[:, order]
    cum = torch.cumsum(evals, 0) / torch.sum(evals).clamp_min(1e-12)
    k = int(torch.searchsorted(cum, torch.tensor([tau], dtype=cum.dtype, device=dev)).item()) + 1
    k = min(k, evecs.shape[1])
    return mu.float().cpu(), evecs[:, :k].float().cpu().contiguous(), k


def score_batch(Fe: torch.Tensor, mus: Sequence[torch.Tensor], Cs: Sequence[torch.Tensor]) -> torch.Tensor:
    """Fe: (B, N, D) -> (B, n_models, N) squared reconstruction residuals ||x - mu - C C^T (x - mu)||^2."""
    outs = []
    with torch.inference_mode():
        for mu, C in zip(mus, Cs):
            d = Fe - mu
            outs.append(((d - (d @ C) @ C.T) ** 2).sum(-1))
    return torch.stack(outs, 1)


# ---------------------------------------------------------------- official post-processing
def _gauss_kernel(ksize: int, sigma: float, dev: torch.device) -> torch.Tensor:
    """cv2.getGaussianKernel(ksize, sigma) for odd ksize and sigma > 0."""
    xs = torch.arange(ksize, dtype=torch.float64, device=dev) - (ksize - 1) / 2.0
    k = torch.exp(-xs ** 2 / (2.0 * sigma ** 2))
    return (k / k.sum()).float()


def postprocess(maps: np.ndarray, dev: torch.device,
                eval_size: Optional[int]) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """Raw (n, G, G) maps -> (image scores (n,), maps at eval_size (n, E, E) or None).

    bilinear upsample to IMG_SIZE (== cv2.INTER_LINEAR) -> GaussianBlur(BLUR_K, BLUR_SIGMA) with reflect-101
    borders -> image score = mean of the top TOP_PCT % pixels (official topk_mean) -> optional antialiased
    resize to eval_size for the pixel metrics.
    """
    S, ks, sig = CFG["IMG_SIZE"], int(CFG["BLUR_K"]), float(CFG["BLUR_SIGMA"])
    blur = ks > 1 and sig > 0
    if blur:
        k = _gauss_kernel(ks, sig, dev)
        kh, kv, r = k.view(1, 1, 1, -1), k.view(1, 1, -1, 1), ks // 2
    top_k = max(1, int(S * S * CFG["TOP_PCT"] / 100.0))
    scores, evals = [], []
    with torch.inference_mode():
        for i in range(0, len(maps), 16):
            x = torch.from_numpy(np.ascontiguousarray(maps[i:i + 16], dtype=np.float32)).to(dev).unsqueeze(1)
            x = F.interpolate(x, size=(S, S), mode="bilinear", align_corners=False)
            if blur:
                x = F.pad(x, (r, r, r, r), mode="reflect")             # == cv2.BORDER_REFLECT_101
                x = F.conv2d(F.conv2d(x, kh), kv)
            scores.append(x.flatten(1).topk(top_k, dim=1).values.mean(1).cpu())
            if eval_size is not None:
                if eval_size != S:
                    x = F.interpolate(x, size=(eval_size, eval_size), mode="bilinear", align_corners=False,
                                      antialias=True)
                evals.append(x.squeeze(1).cpu().numpy())
    return torch.cat(scores).numpy(), (np.concatenate(evals) if eval_size is not None else None)

## 6. Checkpointed pipeline

For every category the planner picks the cheapest valid path:

```
result_<eval_id>.json valid for every regime?        -> skip   (no data, no backbone)
cached maps + GT cache valid (ids, test signature, CRC)? -> metrics only on the CPU (no data, no backbone)
otherwise                                             -> load the dataset; the backbone loads only at the first forward pass
```

GPU stages for one category (each stage resumes from the last finished unit):

1. **A.** Per-image statistics (Σx, Σxxᵀ, n) of each shot image and its 30 rotations. One file per image, written when the image is complete.
2. **B.** One PCA model per (shot, seed), stored as `models.pt` (temporary) together with the chosen shot images.
3. **C.** The test set is scored in chunks of `CHUNK` images, stored in `chunks_<CHUNK>_<test_sig>/`, a folder that sits inside the hash-named fit folder. The same forward pass also accumulates the Regime T statistics, so Regime T costs a single extra pass over the test set.
4. **Commit.** `maps.npy` (float32) is written first, then `meta.json` with the ids, test signature, shape and CRC. The meta file marks completion. Temporary files are deleted afterwards.
5. **Metrics.** `per_item_<eval_id>.parquet`, then `result_<eval_id>.json`, which is the completion marker.

With 2 GPUs, one worker thread per GPU takes categories from a shared queue. Each worker has its own backbone copy and its own DataLoader workers. With 1 GPU, everything runs in the main thread.

In [ ]:
# ============================ 6. PIPELINE (checkpointed, config-fingerprinted) ============================
def atomic_torch_save(obj: Any, path: Path) -> None:
    tmp = _tmp(path)
    torch.save(obj, tmp)
    os.replace(tmp, path)


def safe_torch_load(path: Path) -> Optional[Any]:
    """torch.load that returns None for missing or unreadable (e.g. truncated) files."""
    try:
        return torch.load(path, map_location="cpu", weights_only=False) if Path(path).exists() else None
    except Exception:
        return None


def r_cfgs() -> List[Tuple[int, int]]:
    """(shot, seed) pairs of Regime R, seed-major."""
    return [(int(k), int(s)) for s in CFG["SEEDS"] for k in CFG["SHOTS"]]


def n_aug_for(cat: str) -> int:
    return 0 if cat in CFG["NO_AUG_CATEGORIES"] else int(CFG["N_AUG"])


# ---------------------------------------------------------------- cached maps + results
def commit_maps(regime: str, c: Category, maps: np.ndarray, cfgs: List[Tuple[int, int]],
                ranks: List[int], extra: Optional[Dict[str, Any]] = None) -> None:
    """Write maps.npy first and meta.json (with CRC) last, so a crash can never produce a 'valid' partial cache."""
    d = regime_dir(regime, c.ds, c.cat)
    assert maps.dtype == np.float32 and maps.shape == (len(c.test), len(cfgs), grid(), grid()), maps.shape
    atomic_save_npy(d / "maps.npy", maps)
    meta = dict(regime=regime, id=regime_id(regime), test_sig=c.test_sig, train_sig=c.train_sig,
                cfgs=[list(x) for x in cfgs], ranks=[int(r) for r in ranks], shape=list(maps.shape),
                maps_crc=array_crc(maps), created=time.strftime("%Y-%m-%d %H:%M:%S"), **(extra or {}))
    atomic_write_json(d / "meta.json", meta)


def load_maps(regime: str, ds: str, cat: str,
              test_sig: Optional[str] = None) -> Optional[Tuple[np.ndarray, Dict[str, Any]]]:
    """Return (maps, meta) only if every check passes; otherwise delete the stale files and return None."""
    d = regime_dir(regime, ds, cat)
    meta, f = read_json(d / "meta.json"), d / "maps.npy"
    if meta is None and not f.exists():
        return None
    expect_cfgs = r_cfgs() if regime == "R" else [(0, 0)]
    ok = (meta is not None and f.exists() and meta.get("regime") == regime and meta.get("id") == regime_id(regime)
          and [tuple(int(v) for v in x) for x in meta.get("cfgs", [])] == expect_cfgs)
    ok = ok and (test_sig is None or meta.get("test_sig") == test_sig)
    arr = None
    if ok:
        try:
            arr = np.load(f)
            ok = (arr.dtype == np.float32 and list(arr.shape) == meta["shape"] and arr.shape[2:] == (grid(), grid())
                  and arr.shape[1] == len(meta["cfgs"]) and array_crc(arr) == meta["maps_crc"])
        except Exception:
            ok = False
    if not ok:
        log(f"[{ds}/{cat}] [warn] cached Regime {regime} maps failed validation -> deleted, will be recomputed")
        remove(f)
        remove(d / "meta.json")
        return None
    return arr, meta


def result_path(regime: str, ds: str, cat: str) -> Path:
    return regime_dir(regime, ds, cat) / f"result_{eval_id()}.json"


def load_result(regime: str, ds: str, cat: str) -> Optional[Dict[str, Any]]:
    """A finished result for the current configuration (ids match, per-item table exists, and maps CRC matches)."""
    d = regime_dir(regime, ds, cat)
    r = read_json(result_path(regime, ds, cat))
    if r is None or r.get("ids") != dict(regime=regime, regime_id=regime_id(regime), eval=eval_id()):
        return None
    meta = read_json(d / "meta.json")
    if meta is None or meta.get("maps_crc") != r.get("maps_crc"):
        return None
    if not ((d / f"per_item_{eval_id()}.parquet").exists() or (d / f"per_item_{eval_id()}.csv").exists()):
        return None
    return r


def evaluate(regime: str, ds: str, cat: str, maps: np.ndarray, meta: Dict[str, Any], dev: torch.device) -> None:
    """Official post-processing + all metrics from cached raw maps (CPU work; no backbone, no dataset)."""
    t0 = time.time()
    gt = load_gt_cache(ds, cat)
    assert gt is not None and gt["test_sig"] == meta["test_sig"], f"{ds}/{cat}: GT cache does not match the maps"
    labels = np.asarray(gt["labels"], dtype=np.int64)
    cfgs = [tuple(int(v) for v in x) for x in meta["cfgs"]]
    want_pix = (CFG["PIXEL_METRICS"] or CFG["AUPRO"]) and labels.any()
    masks = gt_masks(ds, cat, CFG["EVAL_SIZE"]) if want_pix else None
    flat = maps.reshape(maps.shape[0], maps.shape[1], -1)
    mx, mn = flat.max(-1), flat.mean(-1)
    runs, rows = [], []
    for j, (k, s) in enumerate(cfgs):
        pix = want_pix and (regime == "T" or CFG["PIXEL_METRICS_SEEDS"] == "all" or s == cfgs[0][1])
        sc, ev = postprocess(maps[:, j], dev, CFG["EVAL_SIZE"] if pix else None)
        m = image_metrics(labels, sc)
        m.update(pixel_metrics(ev, masks, CFG["PIXEL_METRICS"], CFG["AUPRO"]) if pix
                 else {key: NAN for key in PIXEL_KEYS + list(PRO_LIMITS)})
        runs.append(dict(regime=regime, shot=k, seed=s, pca_rank=int(meta["ranks"][j]), **m))
        rows.append(pd.DataFrame(dict(dataset=ds, category=cat, regime=regime, shot=k, seed=s,
                                      rel_path=gt["rel_paths"], label=labels, image_score=sc.astype(np.float64),
                                      map_max_raw=mx[:, j].astype(np.float64),
                                      map_mean_raw=mn[:, j].astype(np.float64))))
    d = regime_dir(regime, ds, cat)
    atomic_save_table(d / f"per_item_{eval_id()}.parquet", pd.concat(rows, ignore_index=True))
    res = dict(dataset=ds, category=cat, regime=regime, n_test=int(len(labels)), n_anomalous=int(labels.sum()),
               runs=runs, seconds=round(time.time() - t0, 1), maps_crc=meta["maps_crc"],
               ids=dict(regime=regime, regime_id=regime_id(regime), eval=eval_id()), eval_cfg=eval_fp())
    atomic_write_json(result_path(regime, ds, cat), res)      # written last = completion marker
    df = pd.DataFrame(runs).groupby("shot")[["img_auroc", "img_ap", "pix_auroc", "aupro_30"]].mean() * 100
    log(f"[{ds}/{cat}] Regime {regime} metrics ({res['seconds']:.0f}s CPU), mean over seeds (%):\n{df.round(2).to_string()}")


# ---------------------------------------------------------------- chunked test scoring (resumable)
class ChunkStore:
    """Per-chunk float32 score files inside a folder that is tied to the fit/T hash AND the test signature."""

    def __init__(self, cdir: Path, n_test: int, n_models: int):
        self.cdir, self.n, self.m, self.ch = cdir, n_test, n_models, int(CFG["CHUNK"])
        self.n_chunks = math.ceil(n_test / self.ch)

    def path(self, ci: int) -> Path:
        return self.cdir / f"chunk_{ci:04d}.npy"

    def shape(self, ci: int) -> Tuple[int, int, int, int]:
        return (min(self.ch, self.n - ci * self.ch), self.m, grid(), grid())

    def load(self, ci: int) -> Optional[np.ndarray]:
        p = self.path(ci)
        if not p.exists():
            return None
        try:
            a = np.load(p)
            if a.dtype == np.float32 and a.shape == self.shape(ci) and np.isfinite(a).all():
                return a
        except Exception:
            pass
        log(f"  [warn] invalid chunk {p.name} -> recomputed")
        remove(p)
        return None

    def pending(self, ci: int) -> bool:
        return self.load(ci) is None

    def any_pending(self) -> bool:
        return any(self.pending(ci) for ci in range(self.n_chunks))

    def assemble(self) -> np.ndarray:
        parts = [self.load(ci) for ci in range(self.n_chunks)]
        assert all(p is not None for p in parts), f"missing chunks in {self.cdir}"
        return np.concatenate(parts)

    def save_info(self, ranks: List[int], **extra: Any) -> None:
        self.cdir.mkdir(parents=True, exist_ok=True)
        atomic_write_json(
            self.cdir / "models_info.json",
            dict(n_test=self.n, n_models=self.m, chunk=self.ch, grid=grid(),
                 ranks=[int(r) for r in ranks], **extra),
        )

    def info(self) -> Optional[Dict[str, Any]]:
        f = self.cdir / "models_info.json"
        if not f.exists():
            return None
        d = read_json(f)
        if d is None or (d.get("n_test"), d.get("n_models"), d.get("chunk"), d.get("grid")) != (
                self.n, self.m, self.ch, grid()):
            log(f"  [warn] stale/invalid chunk folder {self.cdir.name} -> cleared")
            remove(self.cdir)
            return None
        return d


class MapScorer:
    """Test-pass consumer: residual maps of a set of PCA models, saved per chunk."""

    def __init__(self, store: ChunkStore, mus: List[torch.Tensor], Cs: List[torch.Tensor], dev: torch.device):
        self.store = store
        self.mus, self.Cs = [m.to(dev) for m in mus], [C.to(dev) for C in Cs]
        store.cdir.mkdir(parents=True, exist_ok=True)

    def pending(self, ci: int) -> bool:
        return self.store.pending(ci)

    def begin(self, ci: int) -> None:
        self.parts: List[np.ndarray] = []

    def add(self, ci: int, Fe: torch.Tensor) -> None:
        G = grid()
        self.parts.append(score_batch(Fe, self.mus, self.Cs).reshape(len(Fe), len(self.mus), G, G).cpu().numpy())

    def end(self, ci: int) -> None:
        atomic_save_npy(self.store.path(ci), np.concatenate(self.parts).astype(np.float32, copy=False))


class StatsAccumulator:
    """Test-pass consumer: sufficient statistics of the (non-augmented) test features for Regime T."""

    def __init__(self, sdir: Path, n_test: int, tag: Dict[str, Any]):
        self.sdir, self.n, self.tag, self.ch = sdir, n_test, tag, int(CFG["CHUNK"])
        self.n_chunks = math.ceil(n_test / self.ch)
        sdir.mkdir(parents=True, exist_ok=True)

    def path(self, ci: int) -> Path:
        return self.sdir / f"tstat_{ci:04d}.pt"

    def load(self, ci: int) -> Optional[Dict[str, Any]]:
        obj = safe_torch_load(self.path(ci))
        rows = min(self.ch, self.n - ci * self.ch)
        if obj is not None and obj.get("tag") == dict(self.tag, chunk=ci) and obj.get("n") == rows * grid() ** 2:
            return obj
        if self.path(ci).exists():
            remove(self.path(ci))
        return None

    def pending(self, ci: int) -> bool:
        return self.load(ci) is None

    def begin(self, ci: int) -> None:
        self.S1 = self.S2 = None
        self.cnt = 0

    def add(self, ci: int, Fe: torch.Tensor) -> None:
        X = Fe.reshape(-1, Fe.shape[-1]).double()
        if self.S1 is None:
            self.S1 = torch.zeros(X.shape[1], dtype=torch.float64, device=X.device)
            self.S2 = torch.zeros(X.shape[1], X.shape[1], dtype=torch.float64, device=X.device)
        self.S1 += X.sum(0)
        self.S2 += X.T @ X
        self.cnt += X.shape[0]

    def end(self, ci: int) -> None:
        atomic_torch_save(dict(S1=self.S1.cpu(), S2=self.S2.cpu(), n=int(self.cnt), tag=dict(self.tag, chunk=ci)),
                          self.path(ci))

    def total(self) -> Tuple[torch.Tensor, torch.Tensor, int]:
        objs = [self.load(ci) for ci in range(self.n_chunks)]
        assert all(o is not None for o in objs), f"missing Regime T statistics in {self.sdir}"
        return sum(o["S1"] for o in objs), sum(o["S2"] for o in objs), sum(o["n"] for o in objs)


def run_test_pass(dev: torch.device, c: Category, consumers: List[Any], desc: str) -> None:
    """One forward pass over the test chunks that at least one consumer still needs (batches never cross chunks)."""
    n, ch, bs = len(c.test), int(CFG["CHUNK"]), int(CFG["BATCH"])
    todo = {ci: [x for x in consumers if x.pending(ci)] for ci in range(math.ceil(n / ch))}
    todo = {ci: xs for ci, xs in todo.items() if xs}
    if not todo:
        return
    batches = [list(range(b, min(min(n, (ci + 1) * ch), b + bs)))
               for ci in todo for b in range(ci * ch, min(n, (ci + 1) * ch), bs)]
    model = get_model(dev)
    loader = make_loader(ViewDataset([(p, None) for p in c.test_paths], CFG["IMG_SIZE"]), batches, dev)
    bar = tqdm(total=sum(map(len, batches)), desc=f"[{dev}] {c.cat}: {desc}", leave=False)
    cur = None
    for b, xb in zip(batches, loader):
        ci = b[0] // ch
        check_time()
        if ci != cur:
            cur = ci
            for x in todo[ci]:
                x.begin(ci)
        Fe = extract(model, xb, dev)
        for x in todo[ci]:
            x.add(ci, Fe)
        if b[-1] == min(n, (ci + 1) * ch) - 1:
            for x in todo[ci]:
                x.end(ci)
        bar.update(len(b))
    bar.close()


# ---------------------------------------------------------------- Regime R: stage A (statistics) + B (PCA)
def _stat_key(c: Category, i: int) -> str:
    return f"{c.ds}/{c.cat}/{c.rel_train[i]}"


def _stat_path(c: Category, i: int) -> Path:
    return stats_dir(c.ds, c.cat) / f"{i:04d}_{zlib.crc32(_stat_key(c, i).encode()):08x}.pt"


def ensure_train_stats(dev: torch.device, c: Category, pool: List[int]) -> Dict[int, Dict[str, Any]]:
    """Stage A: statistics of every needed shot image (original + rotations), one validated file per image."""
    n_views = 1 + n_aug_for(c.cat)
    expect = dict(feat_id=feat_id(), n_views=n_views, n=n_views * grid() ** 2)
    out: Dict[int, Dict[str, Any]] = {}
    missing = []
    for i in pool:
        obj = safe_torch_load(_stat_path(c, i))
        if obj is not None and obj.get("key") == _stat_key(c, i) and all(obj.get(k) == v for k, v in expect.items()):
            out[i] = obj
        else:
            remove(_stat_path(c, i))
            missing.append(i)
    if not missing:
        return out
    stats_dir(c.ds, c.cat).mkdir(parents=True, exist_ok=True)
    items, batches, owner, last = [], [], [], {}
    bs = int(CFG["BATCH"])
    for i in missing:
        base = len(items)
        items += [(c.train[i], a) for a in rotation_angles(_stat_key(c, i), n_views - 1)]
        for b in range(0, n_views, bs):
            batches.append(list(range(base + b, base + min(n_views, b + bs))))
            owner.append(i)
        last[i] = base + n_views - 1
    model = get_model(dev)
    loader = make_loader(ViewDataset(items, CFG["IMG_SIZE"]), batches, dev)
    bar = tqdm(total=len(missing), desc=f"[{dev}] {c.cat}: shot statistics", leave=False)
    acc: Dict[str, Any] = {}
    for b, i, xb in zip(batches, owner, loader):
        check_time()
        X = extract(model, xb, dev).reshape(-1, model.config.hidden_size).double()
        if acc.get("i") != i:
            acc = dict(i=i, S1=torch.zeros(X.shape[1], dtype=torch.float64, device=X.device),
                       S2=torch.zeros(X.shape[1], X.shape[1], dtype=torch.float64, device=X.device), n=0)
        acc["S1"] += X.sum(0)
        acc["S2"] += X.T @ X
        acc["n"] += X.shape[0]
        if b[-1] == last[i]:
            obj = dict(S1=acc["S1"].cpu(), S2=acc["S2"].cpu(), key=_stat_key(c, i), **dict(expect, n=int(acc["n"])))
            atomic_torch_save(obj, _stat_path(c, i))
            out[i] = obj
            bar.update(1)
    bar.close()
    return out


def load_or_fit_models(dev: torch.device, c: Category) -> Dict[str, Any]:
    """Stage B: one PCA model per (shot, seed). Shots of seed s are the first k of a fixed permutation (nested)."""
    mpath = fit_dir(c.ds, c.cat) / "models.pt"
    cfgs = r_cfgs()
    M = safe_torch_load(mpath)
    if M is not None and M.get("fit_id") == fit_id() and M.get("train_sig") == c.train_sig \
            and [tuple(x) for x in M.get("cfgs", [])] == cfgs:
        return M
    if mpath.exists():
        log(f"[{c.ds}/{c.cat}] [warn] stale/corrupt models.pt -> deleted, will be refitted")
        remove(mpath)
    kmax = max(CFG["SHOTS"])
    assert len(c.train) >= kmax, f"{c.ds}/{c.cat}: only {len(c.train)} training images for {kmax}-shot"
    perms = {s: np.random.RandomState(1000 + s).permutation(len(c.train)) for s in CFG["SEEDS"]}
    pool = sorted({int(i) for s in CFG["SEEDS"] for i in perms[s][:kmax]})
    log(f"[{dev}][{c.ds}/{c.cat}] stage A/B: {len(pool)} unique shot images x {1 + n_aug_for(c.cat)} views "
        f"-> {len(cfgs)} PCA models")
    stats = ensure_train_stats(dev, c, pool)
    mus, Cs, ranks, shots_used = [], [], [], []
    for k, s in cfgs:
        sel = [int(i) for i in perms[s][:k]]
        mu, C, r = fit_pca(sum(stats[i]["S1"] for i in sel), sum(stats[i]["S2"] for i in sel),
                           sum(stats[i]["n"] for i in sel), CFG["TAU"], dev)
        mus.append(mu)
        Cs.append(C)
        ranks.append(r)
        shots_used.append([c.rel_train[i] for i in sel])
    M = dict(fit_id=fit_id(), train_sig=c.train_sig, cfgs=cfgs, mu=mus, C=Cs, rank=ranks, shots_used=shots_used)
    atomic_torch_save(M, mpath)
    return M


def t_stats_consumer(c: Category) -> StatsAccumulator:
    return StatsAccumulator(t_dir(c.ds, c.cat) / f"tstats_{CFG['CHUNK']}_{c.test_sig}", len(c.test),
                            dict(t_id=t_id(), test_sig=c.test_sig))


def t_store(c: Category) -> ChunkStore:
    return ChunkStore(t_dir(c.ds, c.cat) / f"chunks_{CFG['CHUNK']}_{c.test_sig}", len(c.test), 1)


def t_needs_stats(c: Category) -> bool:
    """True if Regime T still needs its PCA statistics (no valid maps, no fitted model, unfinished chunks)."""
    if load_maps("T", c.ds, c.cat, c.test_sig) is not None or load_t_model(c) is not None:
        return False
    s = t_store(c)
    return s.any_pending() or s.info() is None


def load_t_model(c: Category) -> Optional[Dict[str, Any]]:
    p = t_dir(c.ds, c.cat) / "model.pt"
    if not p.exists():
        return None
    M = safe_torch_load(p)
    if M is not None and M.get("t_id") == t_id() and M.get("test_sig") == c.test_sig:
        return M
    log(f"[{c.ds}/{c.cat}] [warn] stale/corrupt Regime T model.pt -> deleted, will be recomputed")
    remove(p)
    return None


def compute_r_maps(dev: torch.device, c: Category, t_acc: Optional[StatsAccumulator]) -> None:
    """Stages A-C + commit for Regime R. Optionally accumulates Regime T statistics in the same test pass."""
    fdir = fit_dir(c.ds, c.cat)
    ensure_fingerprint(CKPT / f"feat_{feat_id()}", feat_fp())
    ensure_fingerprint(fdir.parent.parent, fit_fp())
    fdir.mkdir(parents=True, exist_ok=True)
    if load_maps("R", c.ds, c.cat, c.test_sig) is not None:
        return
    store = ChunkStore(fdir / f"chunks_{CFG['CHUNK']}_{c.test_sig}", len(c.test), len(r_cfgs()))
    consumers: List[Any] = []
    if store.info() is None or store.any_pending():
        M = load_or_fit_models(dev, c)
        store.save_info(M["rank"], shots_used=M["shots_used"])
        consumers.append(MapScorer(store, M["mu"], M["C"], dev))
    if t_acc is not None:
        consumers.append(t_acc)
    run_test_pass(dev, c, consumers, "test scoring (R" + (" + T stats)" if t_acc else ")"))
    info = store.info()
    commit_maps("R", c, store.assemble(), r_cfgs(), info["ranks"], dict(shots_used=info["shots_used"]))
    remove(store.cdir)
    if not CFG["KEEP_MODELS"]:
        remove(fdir / "models.pt")
    if not CFG["KEEP_STATS"]:
        remove(stats_dir(c.ds, c.cat))


def compute_t_maps(dev: torch.device, c: Category) -> None:
    """Regime T (batched zero-shot): PCA on all test features of the category, then score the same images."""
    tdir = t_dir(c.ds, c.cat)
    ensure_fingerprint(tdir.parent.parent, t_fp())
    tdir.mkdir(parents=True, exist_ok=True)
    if load_maps("T", c.ds, c.cat, c.test_sig) is not None:
        return
    store = t_store(c)
    if store.info() is None or store.any_pending():
        Mt = load_t_model(c)
        if Mt is None:
            acc = t_stats_consumer(c)
            run_test_pass(dev, c, [acc], "Regime T statistics")
            S1, S2, n = acc.total()
            mu, C, r = fit_pca(S1, S2, n, CFG["TAU"], dev)
            Mt = dict(t_id=t_id(), test_sig=c.test_sig, mu=mu, C=C, rank=r)
            atomic_torch_save(Mt, tdir / "model.pt")
            remove(acc.sdir)
        store.save_info([Mt["rank"]])
        run_test_pass(dev, c, [MapScorer(store, [Mt["mu"]], [Mt["C"]], dev)], "test scoring (T)")
    commit_maps("T", c, store.assemble(), [(0, 0)], store.info()["ranks"])
    remove(store.cdir)
    if not CFG["KEEP_MODELS"]:
        remove(tdir / "model.pt")


# ---------------------------------------------------------------- per-category driver + scheduler
def needed_regimes(ds: str, cat: str) -> List[str]:
    return [rg for rg in active_regimes() if load_result(rg, ds, cat) is None]


def category_status(ds: str, cat: str) -> str:
    """'done' | 'metrics' (CPU only, from cache) | 'gpu' (dataset needed; backbone loaded only if needed)."""
    need = needed_regimes(ds, cat)
    if not need:
        return "done"
    gt = load_gt_cache(ds, cat)
    if gt is not None and all(load_maps(rg, ds, cat, gt["test_sig"]) is not None for rg in need):
        return "metrics"
    return "gpu"


def process_cached(ds: str, cat: str, dev: torch.device) -> None:
    """Metrics-only path: cached maps + cached ground truth, no dataset and no backbone."""
    sig = load_gt_cache(ds, cat)["test_sig"]
    for rg in needed_regimes(ds, cat):
        maps, meta = load_maps(rg, ds, cat, sig)
        evaluate(rg, ds, cat, maps, meta, dev)


def process_category(dev: torch.device, ds: str, root: Path, cat: str) -> None:
    """Full path for one category (every stage resumes from its own checkpoints)."""
    need = needed_regimes(ds, cat)
    if not need:
        return
    t0 = time.time()
    c = Category(ds, cat, root)
    log(f"[{dev}][{ds}/{cat}] train={len(c.train)} test={len(c.test)} anomalous={int(c.labels.sum())} "
        f"| regimes to finish: {need}")
    save_gt_cache(c)
    if "R" in need:
        want_t = "T" in need and t_needs_stats(c)
        compute_r_maps(dev, c, t_stats_consumer(c) if want_t else None)
    if "T" in need:
        compute_t_maps(dev, c)
    for rg in need:
        maps, meta = load_maps(rg, ds, cat, c.test_sig)
        evaluate(rg, ds, cat, maps, meta, dev)
    log(f"[{dev}][{ds}/{cat}] done in {time.time() - t0:.0f}s")


def run_all() -> str:
    """Plan every category, run metrics-only work on the CPU, then GPU work on 1..MAX_GPUS worker threads.

    Returns 'complete', 'time_budget' or raises the first worker error.
    """
    STOP_EVENT.clear()
    plan: Dict[str, List[Tuple[str, str]]] = {"done": [], "metrics": [], "gpu": []}
    for ds in CFG["DATASETS"]:
        for cat in categories_of(ds):
            plan[category_status(ds, cat)].append((ds, cat))
    log(f"plan: {len(plan['done'])} done | {len(plan['metrics'])} metrics-only (no data, no backbone) "
        f"| {len(plan['gpu'])} need data/GPU")
    dev0 = default_device()
    for ds, cat in plan["metrics"]:
        process_cached(ds, cat, dev0)
    if not plan["gpu"]:
        return "complete"
    roots = {ds: ensure_dataset(ds) for ds in sorted({ds for ds, _ in plan["gpu"]})}   # main thread, once
    tasks: "queue.Queue[Tuple[str, Path, str]]" = queue.Queue()
    for ds, cat in plan["gpu"]:
        tasks.put((ds, roots[ds], cat))
    n_workers = max(1, min(N_CUDA, int(CFG["MAX_GPUS"]), len(plan["gpu"])))
    devices = [torch.device(f"cuda:{i}") for i in range(n_workers)] if N_CUDA else [torch.device("cpu")]
    errors: List[BaseException] = []
    time_up = threading.Event()

    def worker(dev: torch.device) -> None:
        try:
            while not STOP_EVENT.is_set():
                try:
                    ds, root, cat = tasks.get_nowait()
                except queue.Empty:
                    break
                process_category(dev, ds, root, cat)
        except TimeBudgetExceeded:
            time_up.set()
            STOP_EVENT.set()
        except BaseException as e:          # noqa: BLE001 - re-raised in the main thread below
            errors.append(e)
            STOP_EVENT.set()
        finally:
            release_model(dev)

    log(f"GPU work on {len(devices)} worker(s): {[str(d) for d in devices]}")
    if len(devices) == 1:
        worker(devices[0])
    else:
        threads = [threading.Thread(target=worker, args=(d,), daemon=True) for d in devices]
        for t in threads:
            t.start()
        for t in threads:
            t.join()
    if errors:
        raise errors[0]
    return "time_budget" if time_up.is_set() else "complete"


# ---------------------------------------------------------------- session helpers
def restore_checkpoints() -> int:
    """Copy checkpoints of a previous version attached under /kaggle/input (never overwrites, skips temp files)."""
    src_base = Path("/kaggle/input")
    if not (CFG["RESTORE_FROM_INPUT"] and src_base.exists()):
        return 0
    n_copied, base_depth = 0, str(src_base).count(os.sep)
    for root, dirs, _ in os.walk(src_base):
        if root.count(os.sep) - base_depth >= 6:
            dirs[:] = []
            continue
        if os.path.basename(root) == "subspacead_ckpt":
            for r2, _, files in os.walk(root):
                for fn in files:
                    if ".tmp" in fn:
                        continue
                    s = Path(r2) / fn
                    dst = CKPT_ROOT / s.relative_to(root)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(s, dst)
                        n_copied += 1
            dirs[:] = []
    log(f"restored {n_copied} checkpoint files from previous version(s)" if n_copied
        else "no previous checkpoints found under /kaggle/input")
    return n_copied


def write_provenance() -> None:
    """Append one JSON line per session: versions, GPUs, cache ids and the full configuration."""
    info = dict(time=time.strftime("%Y-%m-%d %H:%M:%S"), python=platform.python_version(), torch=torch.__version__,
                numpy=np.__version__, gpus=[torch.cuda.get_device_name(i) for i in range(N_CUDA)],
                ids=dict(feat=feat_id(), fit=fit_id(), T=t_id(), eval=eval_id()), cfg=CFG)
    try:
        import transformers
        info["transformers"] = transformers.__version__
    except Exception:
        pass
    with open(CKPT / "provenance.jsonl", "a") as f:
        f.write(json.dumps(info, default=str) + "\n")

## 7. Run

This cell restores the checkpoints of a previous version, logs provenance, and runs the planner and scheduler. Running it again is always safe. Categories that are already finished cost a few file reads and nothing else.

In [ ]:
# ============================ 7. RUN ============================
restore_checkpoints()
write_provenance()
RUN_STATUS = run_all()
if RUN_STATUS == "time_budget":
    print("\n*** Time budget reached: all finished stages are saved. Save Version, attach this version's output "
          "as input in a new session and Run All to continue. ***")
else:
    print(f"\nALL DONE. Checkpoints: {CKPT}")

## 8. Results tables (CPU only, can be re-run at any time)

This section reads the `result_<eval_id>.json` files of the current configuration and prints three tables:

* per-category metrics (mean over seeds);
* dataset means, computed per seed over categories and then reported as mean ± std over seeds;
* the image AUROC numbers from the official README, for comparison.

CSV files go to `results_<mode>/`, and all per-image scores are combined into one table.

> Only the README image-AUROC values (Regimes R and T) are used as references. The other metrics are reported but **not** compared against the paper. Partial runs (not all categories) are flagged and are not comparable.

In [ ]:
# ============================ 8. RESULTS ============================
# Official README (github.com/CLendering/SubspaceAD), image AUROC (%)
REF_IMG_AUROC = {"R": {"mvtec": {1: 97.1, 2: 97.5, 4: 98.0}, "visa": {1: 93.2, 2: 93.8, 4: 94.7}},
                 "T": {"mvtec": {0: 96.6}, "visa": {0: 94.1}}}
SHOW = ["img_auroc", "img_ap", "pix_auroc", "pix_ap", "aupro_30", "aupro_5"]


def collect_results() -> Optional[pd.DataFrame]:
    """One row per (dataset, category, regime, shot, seed) for the current configuration."""
    rows = []
    for ds in CFG["DATASETS"]:
        for cat in categories_of(ds):
            for rg in active_regimes():
                r = load_result(rg, ds, cat)
                if r is not None:
                    rows += [dict(dataset=ds, category=cat, **run) for run in r["runs"]]
    return pd.DataFrame(rows) if rows else None


def summarize() -> Optional[pd.DataFrame]:
    """Print per-category and dataset-level tables and export CSV / per-item files to RESULTS_DIR."""
    df = collect_results()
    if df is None:
        print("no finished categories yet for the current configuration")
        return None
    df.to_csv(RESULTS_DIR / "all_runs.csv", index=False)
    for (ds, rg), g in df.groupby(["dataset", "regime"]):
        n_done, n_all = g["category"].nunique(), len(categories_of(ds))
        print(f"\n================ {ds.upper()} | Regime {rg} ({n_done}/{n_all} categories) ================")
        per_cat = g.groupby(["shot", "category"])[SHOW].mean().mul(100).round(2)
        per_cat.to_csv(RESULTS_DIR / f"{ds}_regime{rg}_per_category.csv")
        print("Per category, mean over seeds (%):\n" + per_cat.to_string())
        per_seed = g.groupby(["shot", "seed"])[METRIC_KEYS].mean().mul(100)        # mean over categories first
        agg = per_seed.groupby("shot").agg(["mean", "std"]).round(2)
        agg.to_csv(RESULTS_DIR / f"{ds}_regime{rg}_summary.csv")
        print("\nMean over categories, mean/std over seeds (%):\n" + agg[SHOW].to_string())
        ref = REF_IMG_AUROC.get(rg, {}).get(ds)
        if ref:
            print(f"\nOfficial README image AUROC (%): {ref}")
        if n_done < n_all or len(categories_of(ds)) < len(CATS[ds]):
            print("(partial run: not comparable to the paper averages)")
    items = []
    for ds in CFG["DATASETS"]:
        for cat in categories_of(ds):
            for rg in active_regimes():
                for ext in ("parquet", "csv"):
                    f = regime_dir(rg, ds, cat) / f"per_item_{eval_id()}.{ext}"
                    if f.exists():
                        items.append(pd.read_parquet(f) if ext == "parquet" else pd.read_csv(f))
                        break
    if items:
        out = atomic_save_table(RESULTS_DIR / "per_item_all.parquet", pd.concat(items, ignore_index=True))
        print(f"\nper-image scores: {out}")
    return df


RESULTS = summarize()

## 9. Visualisation (optional; cached maps + cached masks)

This uses the same official post-processing as the metrics. The image row is drawn only if the dataset can be found (it is never downloaded for this). The ground-truth and heat-map rows always come from the cache.

In [ ]:
# ============================ 9. VISUALISATION ============================
import matplotlib.pyplot as plt


def show_examples(ds: str, cat: str, regime: str = "R", shot: Optional[int] = None, seed: Optional[int] = None,
                  n: int = 6, rng_seed: int = 0) -> None:
    """Plot n anomalous test images with ground truth and the SubspaceAD map of one (shot, seed) run."""
    gt = load_gt_cache(ds, cat)
    loaded = load_maps(regime, ds, cat, gt["test_sig"] if gt else None)
    if gt is None or loaded is None:
        print(f"{ds}/{cat}: no cached Regime {regime} maps for the current configuration")
        return
    maps, meta = loaded
    cfgs = [tuple(x) for x in meta["cfgs"]]
    shot = (max(CFG["SHOTS"]) if regime == "R" else 0) if shot is None else shot
    seed = cfgs[0][1] if seed is None else seed
    j = cfgs.index((shot, seed))
    labels = np.asarray(gt["labels"])
    idx = np.flatnonzero(labels == 1)
    idx = np.sort(np.random.RandomState(rng_seed).choice(idx, min(n, len(idx)), replace=False))
    sz = CFG["EVAL_SIZE"]
    _, up = postprocess(maps[idx, j], default_device(), sz)
    masks = gt_masks(ds, cat, sz)[idx]
    lo, hi = np.percentile(up, 1), np.percentile(up, 99.9)
    root = find_root(ds)
    if root is not None and not all((Path(root) / gt["rel_paths"][i]).exists() for i in idx):
        root = None
    rows = 3 if root is not None else 2
    fig, ax = plt.subplots(rows, len(idx), figsize=(3 * len(idx), 3 * rows), squeeze=False)
    for col, i in enumerate(idx):
        r = 0
        if root is not None:
            ax[0, col].imshow(Image.open(Path(root) / gt["rel_paths"][i]).convert("RGB").resize((sz, sz)))
            r = 1
        ax[r, col].imshow(masks[col], cmap="gray")
        ax[r + 1, col].imshow(np.clip((up[col] - lo) / (hi - lo + 1e-12), 0, 1), cmap="jet")
        for rr in range(rows):
            ax[rr, col].axis("off")
    title = f"Regime {regime}" + (f" {shot}-shot seed {seed}" if regime == "R" else " (batched 0-shot)")
    ax[0, 0].set_title("image" if root is not None else "ground truth")
    ax[rows - 1, 0].set_title(f"SubspaceAD {title}")
    plt.tight_layout()
    plt.show()


# Example (after a category finished):
# show_examples("mvtec", "bottle", regime="R")
# show_examples("mvtec", "bottle", regime="T")

## 10. Optional: fp16-autocast vs fp32 precision check

This measures how far the default `DTYPE="fp16"` (autocast) features are from the official fp32 features on one real image. It needs the backbone and one dataset image, so it is commented out by default.

In [ ]:
# ============================ 10. OPTIONAL PRECISION CHECK ============================
def precision_check(path: Path, dev: Optional[torch.device] = None) -> Dict[str, float]:
    """Relative L2 error and mean patch cosine of fp16-autocast features vs fp32 features for one image."""
    dev = dev or default_device()
    model = get_model(dev)
    x = torch.from_numpy(load_view(path, CFG["IMG_SIZE"])).unsqueeze(0)
    old = CFG["DTYPE"]
    try:
        CFG["DTYPE"] = "fp16"
        f16 = extract(model, x, dev)
        CFG["DTYPE"] = "fp32"
        f32 = extract(model, x, dev)
    finally:
        CFG["DTYPE"] = old
    out = dict(rel_l2=((f16 - f32).norm() / f32.norm()).item(),
               mean_cos=F.cosine_similarity(f16, f32, dim=-1).mean().item())
    print(f"fp16-autocast vs fp32: relative L2 error = {out['rel_l2']:.5f}, mean patch cosine = {out['mean_cos']:.6f}")
    return out


# _c = Category("mvtec", "bottle", ensure_dataset("mvtec")); precision_check(_c.train[0])